# F1 Data Collection Notebook

This notebook allows you to interactively collect F1 race data using the enhanced FastF1 collector. You can specify the years, session types, and output directories, and inspect the collected data directly in the notebook.

In [ ]:
# Install required packages if not already installed
!pip install fastf1 tqdm pandas

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import fastf1
from tqdm import tqdm

# Add src to path for imports
sys.path.append(str(Path(os.getcwd()).parent / 'src'))
sys.path.append(str(Path(os.getcwd()).parent))

from src.data.fastf1_collector import F1DataCollector

## Configure Data Collection Parameters

In [ ]:
# User parameters
years = [2022, 2023]  # List of years to collect
sessions = ['Q', 'R']  # Session types: FP1, FP2, FP3, Q, R, S
output_dir = 'data/notebook_raw'  # Output directory for collected data
cache_dir = 'data/notebook_cache'  # FastF1 cache directory
save_individual = True  # Save individual event/session files

print(f'Years: {years}')
print(f'Sessions: {sessions}')
print(f'Output directory: {output_dir}')
print(f'Cache directory: {cache_dir}')

In [ ]:
# Initialize the data collector
collector = F1DataCollector(cache_dir=cache_dir)
fastf1.Cache.enable_cache(cache_dir)

## Collect Data for Each Year and Session
This cell will collect data for the specified years and sessions, and save the results to the output directory.

In [ ]:
all_data = []
for year in years:
    print(f'\nCollecting data for {year}...')
    season_data = collector.get_season_data(year, sessions)
    all_data.append(season_data)
    # Save combined data for the season
    season_file = Path(output_dir) / f'season_{year}_data.csv'
    season_file.parent.mkdir(parents=True, exist_ok=True)
    season_data.to_csv(season_file, index=False)
    print(f'Saved season data to {season_file}')
    # Optionally save individual event/session files
    if save_individual:
        for (event, session), event_data in season_data.groupby(['Event', 'Session']):
            event_clean = event.replace(' ', '_')
            event_file = Path(output_dir) / 'events' / f'{year}_{event_clean}_{session}.csv'
            event_file.parent.mkdir(parents=True, exist_ok=True)
            event_data.to_csv(event_file, index=False)
            print(f'Saved {event} {session} data to {event_file}')

# Combine all data
if all_data:
    combined_data = pd.concat(all_data, ignore_index=True)
    combined_file = Path(output_dir) / f'seasons_{"_".join(map(str, years))}_data.csv'
    combined_data.to_csv(combined_file, index=False)
    print(f'Combined data saved to {combined_file}')
else:
    print('No data collected.')

## Inspect Collected Data
You can now load and inspect the collected data.

In [ ]:
# Load the combined data for inspection
combined_file = Path(output_dir) / f'seasons_{"_".join(map(str, years))}_data.csv'
if combined_file.exists():
    df = pd.read_csv(combined_file)
    display(df.head())
    print(f'Total records: {len(df)}')
else:
    print('Combined data file not found.')